<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/10_clip_based_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 — Phase 10: CLIP-Based Detector (Foundation Model Adapter)

Tests whether a frozen CLIP vision backbone — pretrained on web-scale
image-text data, far broader than ImageNet — generalizes better **zero-shot**
to unseen forgery types than a CNN (ResNet18) trained from scratch on narrow
data (SOTA review, Section 4.4).

**Design, for a clean comparison:** trained on StyleGAN only, with the exact
same training data as Phase 2, so this experiment isolates the effect of the
backbone/pretraining (CLIP vs. ImageNet-pretrained ResNet18) as the one
variable of interest. Only a small linear classification head is trained on
top of CLIP's frozen image embeddings — CLIP's own weights are never updated
("linear probing" an adapter, as described in the SOTA review). Evaluated
zero-shot on face-swap and diffusion-generated faces, directly against
Phase 2's numbers (StyleGAN 0.999 AUC, Face-swap 0.525 AUC, Diffusion 0.277 AUC).

Because CLIP stays frozen, this notebook is computationally lighter than
earlier ones: embeddings are extracted once (no backpropagation through CLIP
itself), then the small head trains very fast on the cached embeddings.


### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

Mon Oct  5 09:40:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to / loaded from:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to / loaded from: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle API access (via Colab Secrets)


In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_KEY_HERE"

### Step 4 — Download all three datasets (training uses StyleGAN only; the other two are for zero-shot evaluation)

In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d data_raw

!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q deepfake-and-real-images.zip -d data_raw_2

!kaggle datasets download -d bwandowando/faces-dataset-using-stable-diffusion-v14
!unzip -q faces-dataset-using-stable-diffusion-v14.zip -d data_diffusion
print("All three datasets downloaded.")

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:40<00:00, 100MB/s] 

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:16<00:00, 107MB/s]

Dataset URL: https://www.kaggle.com/datasets/bwandowando/faces-dataset-using-stable-diffusion-v14
License(s): CC0-1.0
100% 1.10G/1.10G [00:13<00:00, 86.7MB/s]

All three datasets downloaded.


### Step 5 — Locate folders in all three datasets

In [ ]:
def find_real_fake_folders(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        lower = [d.lower() for d in dirnames]
        if "real" in lower and "fake" in lower:
            candidates.append(dirpath)
    return candidates

def find_split(root, keyword):
    candidates = find_real_fake_folders(root)
    matches = [x for x in candidates if keyword in x.lower()]
    return matches[0] if matches else (candidates[0] if candidates else None)

def find_class_dir(split_root, class_name):
    for d in os.listdir(split_root):
        if d.lower() == class_name:
            return os.path.join(split_root, d)
    raise FileNotFoundError(f"No '{class_name}' folder in {split_root}")

def find_all_images(root, exts=(".jpg", ".jpeg", ".png")):
    found = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(exts):
                found.append(os.path.join(dirpath, fname))
    return found

styleGAN_train = find_split("data_raw", "train")
styleGAN_val = find_split("data_raw", "val") or find_split("data_raw", "valid")
styleGAN_test = find_split("data_raw", "test")
faceswap_test = find_split("data_raw_2", "test")
diffusion_images = find_all_images("data_diffusion")

print("StyleGAN train:", styleGAN_train)
print("StyleGAN val:  ", styleGAN_val)
print("StyleGAN test: ", styleGAN_test)
print("Face-swap test:", faceswap_test)
print("Diffusion images found:", len(diffusion_images))

StyleGAN train: data_raw/real_vs_fake/real-vs-fake/train
StyleGAN val:   data_raw/real_vs_fake/real-vs-fake/valid
StyleGAN test:  data_raw/real_vs_fake/real-vs-fake/test
Face-swap test: data_raw_2/Dataset/Test
Diffusion images found: 2536


### Step 6 — Load the frozen CLIP backbone

Uses CLIP ViT-B/32 (via Hugging Face `transformers`), a widely used,
moderately sized CLIP variant that balances representation quality with
practical runtime on a single T4 GPU. All CLIP parameters are frozen —
only a small classification head will be trained later.

In [ ]:
!pip install -q transformers

import torch
from transformers import CLIPModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

embedding_dim = clip_model.config.projection_dim
print(f"CLIP loaded. Image embedding dimension: {embedding_dim}")
print("All CLIP parameters frozen — only a small head will be trained.")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

CLIP loaded. Image embedding dimension: 512
All CLIP parameters frozen — only a small head will be trained.


### Step 7 — Define CLIP's expected image preprocessing

CLIP was trained with its own specific resizing and normalization statistics
(different from ImageNet's), so we replicate that exactly rather than reusing
the ResNet preprocessing from earlier notebooks.

In [ ]:
from torchvision import transforms

clip_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711],
    ),
])
print("CLIP preprocessing transform ready.")

CLIP preprocessing transform ready.


### Step 8 — Extract CLIP embeddings for the StyleGAN training and validation sets

Since CLIP is frozen, its output for a given image never changes — so instead
of running images through CLIP on every training epoch (slow and wasteful),
we run every image through CLIP **once**, cache the resulting embeddings, and
train the small head on those cached embeddings very quickly afterward.
This is the standard, efficient way to linear-probe a frozen foundation model.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets
from tqdm import tqdm

def extract_embeddings(image_folder_dataset, batch_size=128):
    loader = DataLoader(image_folder_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    all_embeddings, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="extracting CLIP embeddings"):
            images = images.to(device)
            # Version-proof: vision tower -> pooled output -> projection to 512-d
            vision_out = clip_model.vision_model(pixel_values=images)
            features = clip_model.visual_projection(vision_out.pooler_output)
            all_embeddings.append(features.cpu())
            all_labels.append(labels)
    return torch.cat(all_embeddings), torch.cat(all_labels)


train_set = datasets.ImageFolder(styleGAN_train, transform=clip_transform)
val_set = datasets.ImageFolder(styleGAN_val, transform=clip_transform)
class_to_idx = train_set.class_to_idx
print("Class mapping:", class_to_idx)

train_embeddings, train_labels = extract_embeddings(train_set)
val_embeddings, val_labels = extract_embeddings(val_set)
print("Train embeddings:", train_embeddings.shape, " Val embeddings:", val_embeddings.shape)

Class mapping: {'fake': 0, 'real': 1}


extracting CLIP embeddings: 100%|██████████| 157/157 [01:22<00:00,  1.90it/s]

Train embeddings: torch.Size([100000, 512])  Val embeddings: torch.Size([20000, 512])


### Step 9 — Train a small linear head on the cached embeddings

A single linear layer (embedding_dim -> 2) is trained — this is the "adapter"
in the SOTA review's terminology: CLIP's representation stays fixed, only
this small layer learns to map it to a real/fake decision. Training on cached
embeddings (not raw images) means this runs in seconds per epoch, not minutes.

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset

head = nn.Linear(embedding_dim, 2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)

train_ds = TensorDataset(train_embeddings, train_labels)
val_ds = TensorDataset(val_embeddings, val_labels)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

best_val_acc = 0.0
n_epochs = 15  # cheap per-epoch cost allows more epochs than earlier full-network training

for epoch in range(n_epochs):
    head.train()
    train_correct, train_total, train_loss_sum = 0, 0, 0.0
    for emb, labels in train_loader:
        emb, labels = emb.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = head(emb)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * emb.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += emb.size(0)

    head.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for emb, labels in val_loader:
            emb, labels = emb.to(device), labels.to(device)
            logits = head(emb)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_total += emb.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/{n_epochs}  train_loss={train_loss_sum/train_total:.4f}  "
          f"train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({"head_state": head.state_dict(), "class_to_idx": class_to_idx},
                    f"{CHECKPOINT_DIR}/best_model_clip_head.pth")

print(f"\nDone. Best validation accuracy: {best_val_acc:.4f}")
print(f"Head checkpoint saved to: {CHECKPOINT_DIR}/best_model_clip_head.pth")

Epoch 1/15  train_loss=0.3531  train_acc=0.8581  val_acc=0.9020
Epoch 2/15  train_loss=0.2348  train_acc=0.9125  val_acc=0.9213
Epoch 3/15  train_loss=0.2039  train_acc=0.9243  val_acc=0.9276
Epoch 4/15  train_loss=0.1880  train_acc=0.9296  val_acc=0.9331
Epoch 5/15  train_loss=0.1779  train_acc=0.9324  val_acc=0.9363
Epoch 6/15  train_loss=0.1710  train_acc=0.9346  val_acc=0.9375
Epoch 7/15  train_loss=0.1657  train_acc=0.9372  val_acc=0.9389
Epoch 8/15  train_loss=0.1616  train_acc=0.9384  val_acc=0.9396
Epoch 9/15  train_loss=0.1583  train_acc=0.9394  val_acc=0.9404
Epoch 10/15  train_loss=0.1557  train_acc=0.9404  val_acc=0.9411
Epoch 11/15  train_loss=0.1533  train_acc=0.9413  val_acc=0.9410
Epoch 12/15  train_loss=0.1515  train_acc=0.9417  val_acc=0.9427
Epoch 13/15  train_loss=0.1499  train_acc=0.9422  val_acc=0.9432
Epoch 14/15  train_loss=0.1485  train_acc=0.9432  val_acc=0.9435
Epoch 15/15  train_loss=0.1472  train_acc=0.9435  val_acc=0.9440

Done. Best validation accuracy: 0

### Step 10 — Build the three zero-shot evaluation sets

Same three domains used throughout this project. Since this model never
trained on any face-swap or diffusion images (unlike Phase 9), the full
diffusion set (2,536 images) can be used here, matching Phase 7/8's original
evaluation scale.

In [ ]:
import random

def build_eval_subset_imagefolder(source_root, dst_root, n_per_class, seed=99):
    rng = random.Random(seed)
    for cls in ["real", "fake"]:
        dst = os.path.join(dst_root, cls)
        os.makedirs(dst, exist_ok=True)
        cls_dir = find_class_dir(source_root, cls)
        files = os.listdir(cls_dir)
        rng.shuffle(files)
        for fname in files[:n_per_class]:
            src = os.path.abspath(os.path.join(cls_dir, fname))
            dst_path = os.path.join(dst, fname)
            if not os.path.exists(dst_path):
                os.symlink(src, dst_path)

build_eval_subset_imagefolder(styleGAN_test, "eval_styleGAN_clip", n_per_class=5000)
build_eval_subset_imagefolder(faceswap_test, "eval_faceswap_clip", n_per_class=5000)

real_dir = find_class_dir(styleGAN_test, "real")
real_images_pool = [os.path.join(real_dir, f) for f in os.listdir(real_dir)]

def build_eval_diffusion(dst_root, fake_paths, real_paths, n_per_class, seed=7):
    rng = random.Random(seed)
    fake_dst = os.path.join(dst_root, "fake")
    real_dst = os.path.join(dst_root, "real")
    os.makedirs(fake_dst, exist_ok=True)
    os.makedirs(real_dst, exist_ok=True)
    fake_sample = fake_paths[:]; rng.shuffle(fake_sample); fake_sample = fake_sample[:n_per_class]
    real_sample = real_paths[:]; rng.shuffle(real_sample); real_sample = real_sample[:n_per_class]
    for i, path in enumerate(fake_sample):
        dst = os.path.join(fake_dst, f"{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)
    for i, path in enumerate(real_sample):
        dst = os.path.join(real_dst, f"{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)

build_eval_diffusion("eval_diffusion_clip", diffusion_images, real_images_pool, n_per_class=5000)
print("All three zero-shot evaluation sets built.")

All three zero-shot evaluation sets built.


### Step 11 — Evaluate the CLIP-based detector on all three domains

Compare directly against Phase 2's RGB/ResNet18 numbers: StyleGAN 98.8% acc /
0.999 AUC, Face-swap 49.8% acc / 0.525 AUC, Diffusion 65.5% acc / 0.277 AUC.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

checkpoint = torch.load(f"{CHECKPOINT_DIR}/best_model_clip_head.pth", map_location=device)
head.load_state_dict(checkpoint["head_state"])
head.eval()
real_idx = checkpoint["class_to_idx"]["real"]


def evaluate_domain(test_dir):
    test_set = datasets.ImageFolder(test_dir, transform=clip_transform)
    embeddings, labels_raw = extract_embeddings(test_set, batch_size=128)
    remap = {idx: checkpoint["class_to_idx"][name.lower()] for name, idx in test_set.class_to_idx.items()}
    labels = torch.tensor([remap[l.item()] for l in labels_raw])

    with torch.no_grad():
        logits = head(embeddings.to(device))
        probs = torch.softmax(logits, dim=1)[:, real_idx].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(labels.numpy(), preds)
    auc = roc_auc_score(labels.numpy(), probs)
    cm = confusion_matrix(labels.numpy(), preds)
    return acc, auc, cm


print(f"{'Domain':<20} {'Accuracy':<12} {'AUC':<10}")
for name, test_dir in [("StyleGAN", "eval_styleGAN_clip"),
                         ("Face-swap", "eval_faceswap_clip"),
                         ("Diffusion", "eval_diffusion_clip")]:
    acc, auc, cm = evaluate_domain(test_dir)
    print(f"{name:<20} {acc:<12.4f} {auc:<10.4f}")
    print(f"  Confusion matrix:\n{cm}\n")

print("\n--- Reference: Phase 2 (ResNet18, ImageNet-pretrained, single-source) ---")
print("StyleGAN:  Accuracy=0.9870  AUC=0.999")
print("Face-swap: Accuracy=0.4979  AUC=0.525")
print("Diffusion: Accuracy=0.655   AUC=0.277")

Domain               Accuracy     AUC       


extracting CLIP embeddings: 100%|██████████| 79/79 [00:49<00:00,  1.60it/s]


StyleGAN             0.9436       0.9878    
  Confusion matrix:
[[4725  275]
 [ 289 4711]]



extracting CLIP embeddings: 100%|██████████| 79/79 [00:44<00:00,  1.79it/s]


Face-swap            0.4967       0.4839    
  Confusion matrix:
[[2708 2292]
 [2741 2259]]



extracting CLIP embeddings: 100%|██████████| 59/59 [01:07<00:00,  1.14s/it]

Diffusion            0.6277       0.2324    
  Confusion matrix:
[[  19 2517]
 [ 289 4711]]


--- Reference: Phase 2 (ResNet18, ImageNet-pretrained, single-source) ---
StyleGAN:  Accuracy=0.9870  AUC=0.999
Face-swap: Accuracy=0.4979  AUC=0.525
Diffusion: Accuracy=0.655   AUC=0.277


---
### What to save for this phase
- CLIP embedding dimension and setup confirmation (Step 6)
- Training curve on cached embeddings (Step 9) — note the much faster
  per-epoch time compared to earlier full-network training notebooks
- The full three-domain accuracy/AUC table from Step 11, compared directly
  against the Phase 2 reference numbers printed at the end

**How to interpret the result:**
- If CLIP's Face-swap and/or Diffusion AUC are meaningfully higher than
  Phase 2's (0.525 and 0.277), that supports the hypothesis that CLIP's
  broader pretraining yields representations that generalize better
  zero-shot than an ImageNet-pretrained CNN's.
- If performance is similar, that's also informative — it would suggest
  the generalization problem is more about the narrowness of the
  *fine-tuning* data than the backbone's pretraining breadth, reinforcing
  the Phase 9 finding that direct data exposure is the dominant factor.

Send me the full output and we'll add this as the next section of your report.